# Atlas fix-plan example

This notebook uses a tiny synthetic Atlas dataset and a JSON fix plan to show the public Woodpecker plan API with bundled plugin fixes.

In [ ]:
from pathlib import Path

import woodpecker_atlas_plugin  # noqa: F401 - imports plugin fixes for editable installs

import woodpecker
from woodpecker.testing import make_atlas

Create a deterministic Atlas-like dataset with two issues: missing `project_id` metadata and overly strong compression settings.

In [ ]:
dataset = make_atlas(missing=["project_id"], seed=7)
dataset["pr"].encoding["complevel"] = 5

dataset

Load the Atlas fix-plan document. It matches `atlas*.nc`-style inputs and selects the bundled Atlas plugin fixes.

In [ ]:
plan_path = Path("notebooks/atlas_basic_plan.json")
if not plan_path.exists():
    plan_path = Path("atlas_basic_plan.json")

plan_path

In [ ]:
findings = woodpecker.check_plan(plan_path, inputs=dataset)
findings.fix_ids

A dry run reports what would change but leaves the dataset untouched.

In [ ]:
dry_run = woodpecker.fix_plan(plan_path, inputs=dataset, write=False)

dry_run.stats, dataset.attrs.get("project_id"), dataset["pr"].encoding["complevel"]

Apply the same plan in memory with `write=True`.

In [ ]:
write = woodpecker.fix_plan(plan_path, inputs=dataset, write=True)

(
    write.stats,
    dataset.attrs["project_id"],
    dataset["pr"].encoding["complevel"],
    dataset["pr"].encoding["zlib"],
    dataset["pr"].encoding["shuffle"],
)

In [ ]:
recheck = woodpecker.check_plan(plan_path, inputs=dataset)
recheck.has_findings